# Film Db Rag

Generated from stack spec (Film DB RAG). This notebook mirrors the RAG Playground flow.

In [ ]:
import os, json, requests
from pathlib import Path

# Optionally load .env written by initContainer into /home/jovyan/work/.rag/.env
envfile = Path('/home/jovyan/work/.rag/.env')
if envfile.exists():
    for line in envfile.read_text().splitlines():
        if '=' in line and not line.strip().startswith('#'):
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

PROXY_HOST = os.environ.get('PROXY_HOST', '')
PROXY_BASE = os.environ.get('PROXY_BASE') or (f'https://{PROXY_HOST}' if PROXY_HOST else '')
SCHEMA_NAME = os.environ.get('SCHEMA_NAME', 'public')

STORAGE_ACCESS_KEY_ID = os.environ.get('STORAGE_ACCESS_KEY_ID', '')
STORAGE_SECRET_ACCESS_KEY = os.environ.get('STORAGE_SECRET_ACCESS_KEY', '')
LLMAPI_BASE = os.environ.get('LLMAPI_BASE', 'https://llmapi6.llmosaic.ai')
EMBED_NAME = os.environ.get('EMBED_NAME', 'titan-embed-text-v2')
EMBED_MODEL = os.environ.get('EMBED_MODEL', 'titan-embed-text-v2')
LLM_NAME = os.environ.get('LLM_NAME', 'gpt-oss-120b')
LLM_MODEL = os.environ.get('LLM_MODEL', 'gpt-oss-120b')
LLMAPI_API_KEY = os.environ.get('LLMAPI_API_KEY', '')

assert PROXY_BASE, 'Set PROXY_BASE or PROXY_HOST in env/.env'
AUTH_BEARER = f'{STORAGE_ACCESS_KEY_ID}:{STORAGE_SECRET_ACCESS_KEY}:storage' if STORAGE_ACCESS_KEY_ID and STORAGE_SECRET_ACCESS_KEY else ''
PROXY_HEADERS = {'Authorization': f'Bearer {AUTH_BEARER}'} if AUTH_BEARER else {}
LLM_HEADERS = {'Authorization': f'Bearer {LLMAPI_API_KEY}', 'Content-Type': 'application/json'}
print('Using PROXY_BASE=', PROXY_BASE)
print('Using SCHEMA_NAME=', SCHEMA_NAME)

## Health Check

Verify PostgREST-Proxy is ready.

In [ ]:
url = PROXY_BASE + '/healthz'
r = requests.get(url, headers=PROXY_HEADERS)
print(r.status_code)
print(r.text[:2000])

## Restore Film DB

Restore the database from the preloaded SQL fixture.

In [ ]:
url = PROXY_BASE + '/restore-fixture?fixture=/artifacts/film_db_backup.sql'
r = requests.get(url, headers=dict(**PROXY_HEADERS, **{'Accept-Profile': SCHEMA_NAME}))
print(r.status_code)
print(r.text[:2000])

## Prepare Film Texts

Extract a small set of film descriptions for embedding.

In [ ]:
url = PROXY_BASE + '/film_list?limit=10'
r = requests.get(url, headers=dict(**PROXY_HEADERS, **{'Accept-Profile': SCHEMA_NAME}))
print(r.status_code)
try:
    j = r.json()
    FILM_DOCS = [row.get('description', '') for row in (j or [])]
    FILM_IDS = [row.get('id') for row in (j or [])]
    print('prepared', len(FILM_DOCS), 'film docs')
except Exception as e:
    print('failed to parse film list:', e)

## Drop Embeddings Table (if exists)

In [ ]:
url = PROXY_BASE + '/drop-table?schemaName=' + SCHEMA_NAME
body = {"table_name": "film_embeddings", "if_exists": True}
r = requests.post(url, headers=dict(**PROXY_HEADERS, **{'Content-Type':'application/json'}), json=body)
print(r.status_code, r.text[:200])

## Create Embeddings Table

In [ ]:
url = PROXY_BASE + '/create-table?schemaName=' + SCHEMA_NAME
body = {
  'table_name': 'film_embeddings',
  'not_exists': True,
  'columns': [
    { 'name': 'id', 'type': 'bigserial', 'constraints': 'PRIMARY KEY' },
    { 'name': 'film_id', 'type': 'integer' },
    { 'name': 'document_text', 'type': 'text' },
    { 'name': 'embedding', 'type': 'vector(1024)' }
  ]
}
r = requests.post(url, headers=dict(**PROXY_HEADERS, **{'Content-Type':'application/json'}), json=body)
print(r.status_code, r.text[:200])

## Create Vector Index (HNSW)

In [ ]:
url = PROXY_BASE + '/create-vector-index?schemaName=' + SCHEMA_NAME
body = {
  'table_name': 'film_embeddings',
  'vector_column': 'embedding',
  'index_type': 'hnsw',
  'distance_operator': 'vector_cosine_ops'
}
r = requests.post(url, headers=dict(**PROXY_HEADERS, **{'Content-Type':'application/json'}), json=body)
print(r.status_code, r.text[:200])

## Embed + Insert Films

In [ ]:
assert 'FILM_DOCS' in globals(), 'Run Prepare Film Texts first'
for i, t in enumerate(FILM_DOCS):
    fid = (FILM_IDS[i] if 'FILM_IDS' in globals() and i < len(FILM_IDS) else i+1)
    er = requests.post(LLMAPI_BASE + '/' + EMBED_NAME + '/v1/embeddings', headers=LLM_HEADERS, json={'model': EMBED_MODEL, 'input': [t]})
    ej = er.json(); vec = (ej.get('data') or [{}])[0].get('embedding')
    ir = requests.post(PROXY_BASE + '/film_embeddings', headers=dict(**PROXY_HEADERS, **{'Content-Type':'application/json','Content-Profile': SCHEMA_NAME}), json={'film_id': fid, 'document_text': t, 'embedding': vec})
    print('insert', fid, ir.status_code)

## Vector Query

In [ ]:
qtext = 'Adventure on the open sea with pirates'
er = requests.post(LLMAPI_BASE + '/' + EMBED_NAME + '/v1/embeddings', headers=LLM_HEADERS, json={'model': EMBED_MODEL, 'input': [qtext]})
ej = er.json(); vec = (ej.get('data') or [{}])[0].get('embedding')
import urllib.parse
encoded = urllib.parse.quote(json.dumps(vec))
qr = requests.get(PROXY_BASE + '/film_embeddings?query_vector=' + encoded + '&vector_column=embedding&distance_operator=<=>&limit=3', headers=dict(**PROXY_HEADERS, **{'Accept-Profile': SCHEMA_NAME}))
print(qr.status_code)
print(qr.text)
try:
    LAST_RESULTS = qr.json()
except Exception:
    LAST_RESULTS = []


## Chat with Retrieved Context

In [ ]:
ctx = ''
try:
    if isinstance(LAST_RESULTS, list) and LAST_RESULTS:
        first = LAST_RESULTS[0]
        ctx = first.get('data', {}).get('text') or first.get('document_text') or ''
except Exception:
    pass
prompt = f"Using the following film synopsis as context: '{ctx}', answer the question: 'Which characters are central to the story?'"
cr = requests.post(LLMAPI_BASE + '/' + LLM_NAME + '/v1/chat/completions', headers=LLM_HEADERS, json={'model': LLM_MODEL, 'messages': [{'role': 'user', 'content': prompt}], 'max_tokens': 256, 'temperature': 0.7})
print(cr.status_code)
print(cr.text[:2000])
